## Scenario

You are a data analyst at a sports news website! You are responsible for gathering sports statistics. The company requires automated scripts to periodically update game data. You will use a sports data API to gather scores and automate their delivery to your desktop.

## Objective

This activity combines API requests, desktop notifications, and task scheduling. The notebook keeps these steps separate so you can see what each library does.

## Configuración segura

Copia `Automatizacion/Modulo_3/.env.ejemplo` como `.env` en la raíz del proyecto y reemplaza el valor de `FOOTBALL_API_KEY` por tu clave. `.env` está excluido de Git; no pegues la clave en una celda ni compartas el archivo. Reinicia el kernel después de crear o cambiar `.env`.

`plyer` solo solicita una notificación al sistema operativo. `requests` obtiene los datos de fútbol y `schedule` decide cuándo ejecutar la función.

In [1]:
import os
import time

import requests
import schedule
from dotenv import load_dotenv
from plyer import notification

load_dotenv()

True

In [2]:
api_key = os.getenv("FOOTBALL_API_KEY")

if api_key:
    print("Football-data.org API key loaded (value hidden).")
else:
    print("No API key found. Add FOOTBALL_API_KEY to the project's .env file.")

Football-data.org API key loaded (value hidden).


In [4]:
API_URL = "https://api.football-data.org/v4/matches"


def fetch_scores():
    if not api_key:
        raise RuntimeError("Configura FOOTBALL_API_KEY en .env antes de consultar la API.")

    response = requests.get(
        API_URL,
        headers={"X-Auth-Token": api_key},
        timeout=15,
    )
    response.raise_for_status()
    data = response.json()
    matches = data.get("matches") if isinstance(data, dict) else None

    if not isinstance(matches, list):
        raise ValueError("La respuesta de la API no contiene una lista 'matches'.")
    if not matches:
        return "No matches currently available."

    lines = []
    for match in matches[:5]:
        home = (match.get("homeTeam") or {}).get("name", "Home team")
        away = (match.get("awayTeam") or {}).get("name", "Away team")
        full_time = ((match.get("score") or {}).get("fullTime") or {})
        home_score = full_time.get("home")
        away_score = full_time.get("away")

        if home_score is None or away_score is None:
            lines.append(f"{home} vs {away} ({match.get('status', 'score unavailable')})")
        else:
            lines.append(f"{home} {home_score} - {away_score} {away}")

    return "\n".join(lines)


# Prueba la consulta real una vez configurada la clave:
print(fetch_scores())

CD Santa Clara 1 - 2 Sport Lisboa e Benfica
Paris Saint-Germain FC 2 - 0 Olympique Lyonnais
Sevilla FC 2 - 0 CA Osasuna
Coritiba FBC 0 - 0 Grêmio FBPA
CR Flamengo 0 - 0 CA Mineiro


In [9]:
def send_notification():
    scores = fetch_scores()
    notification.notify(
        title="Latest football scores",
        message=scores[:250],
        timeout=10,
    )
    print("Desktop notification requested.")
    return scores


# Descomenta para solicitar una notificación ahora:
send_notification()

Desktop notification requested.


'CD Santa Clara 1 - 2 Sport Lisboa e Benfica\nParis Saint-Germain FC 0 - 0 Olympique Lyonnais\nSevilla FC 0 - 0 CA Osasuna\nCoritiba FBC vs Grêmio FBPA (TIMED)\nCR Flamengo vs CA Mineiro (TIMED)'

In [11]:
# Evita registrar el mismo trabajo varias veces si vuelves a ejecutar esta celda.
schedule.clear()
schedule.every().day.at("17:29").do(send_notification)
print("Scheduled: send_notification() every day at 17:15.")


def run_scheduler():
    while True:
        schedule.run_pending()
        time.sleep(1)


# Descomenta para dejar el kernel esperando y ejecutar el trabajo cuando llegue la hora:
run_scheduler()

Scheduled: send_notification() every day at 17:15.
Desktop notification requested.


KeyboardInterrupt: 

Pregunta 1
## ¿Qué biblioteca se utiliza para programar la ejecución automatizada de tareas, como el envío de notificaciones a intervalos específicos? Seleccione la mejor respuesta.

- solicita
- tiempo
- **schedule**
- plyer

Buen trabajo
> Correcto Esta biblioteca permite programar tareas para que se ejecuten a intervalos u horas específicos.

Pregunta 2
## ¿Qué tipo de solicitud realiza la función fetch_scores() en esta actividad? Seleccione la mejor respuesta.


- PATCH
- PUT
- **GET**
- POST

> Correcto Esta es la funcionalidad principal de la función fetch_scores(). Realiza una llamada a la API para recuperar los datos mediante una solicitud GET.

## ¿Cómo cambiaría el código para enviar notificaciones cada hora en lugar de cada día a las 8:00 AM? Seleccione la mejor respuesta.


- Sustituya schedule.every().day.at("08:00") por schedule.every().hour.at("08:00")
- **Sustituya schedule.every().day.at("08:00") por schedule.every().hour**
- Sustituya while True: por while False:
- Sustituya schedule.every().day.at("08:00") por schedule.every(8).hour

Buen trabajo

> Correcto! schedule.every().hour establece que el trabajo se ejecute cada hora.

## Si tuviera que cambiar la fuente de football-data.org por otra fuente, ¿qué parámetros tendría que cambiar? Seleccione todo lo que corresponda.


- **La lógica para guardar los datos en score_line**

Buen trabajo

> Correcto Es probable que el formato de los datos en otro sitio sea diferente, lo que obligaría a revisar la lógica.

- Tipo de solicitud

- **La clave API**

> Correcto Su clave API para otra fuente de datos será diferente.

- **La URL**

Buen trabajo
> Correcto La URL tendría que cambiar para reflejar la nueva fuente de datos.

## ¿Qué hace la siguiente línea de código? Seleccione la mejor respuesta.
### schedule.every().day.at("08:00").do(send_notification)


- Envía una notificación todos los días a las 8:00 AM
- Envía una notificación sólo los lunes a las 8:00 AM
- Envía una notificación cada 8 segundos
- Envía una notificación inmediatamente cuando se ejecuta el script

Buen trabajo
> Correcto El código utiliza schedule.every().day.at("08:00") para programar la tarea.